# 59 새 질문에서 선택 설정 확인

58번의 설정과 평가를 먼저 동결하고, 그 다음 새로운32질문 및 원문 정답 기준을 만든다. 설정 선택 전에는 새 질문·정답 파일을 만들지 않는다.

추천24질문은 단일혜택12·복수혜택8·발급사/실적4, 상세8질문은 실적/한도/횟수3·예외/채널2·계산1·부정/확인불가2다. 32질문의 조건 추출을3회씩96번 수행하고, 두 순차검색 프로필에서 같은32질문을3회씩192번 답변한다. 선택된 지침과 프로필별 A/B 근거 설정을 유지하고 유리한 회차로 교체하지 않는다. 질문 임베딩은32개를 한 배치로 보낸다.

모델과 스키마 및 출력2400은 고정하며 단계비용2달러·46번 이후 누적8달러 상한을 적용한다. 새 사용량 미확인 오류는 중단한다. 답변 완료시간은 검증 재시도를 포함해 기록하며 첫 토큰·화면 출력·실제 온라인 전체 응답시간은 미측정이면 표시한다.

같은106문서의 새 질문 확인으로, 새 문서 검증이나 운영 개선량 비교가 아니다. 새 질문의 비구조화 단독 기준선을 추가하지 않으므로 운영 적용 완료나 SLA 통과를 주장하지 않는다.

공유 `.py`는 원시 응답 보존, 중단 재개, 한 번의 임베딩 배치, 설정/질문 생성 순서와 API 예산 계약을 담당한다. HTML은 실제 결과와 원문을 확인하고 직접 작성한다.

In [1]:
from pathlib import Path
import importlib.util,sys,json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'notebooks/scripts').is_dir())
spec=importlib.util.spec_from_file_location('runtime59_notebook',ROOT/'notebooks/scripts/59_confirmation_runtime.py')
N=importlib.util.module_from_spec(spec);sys.modules[spec.name]=N;spec.loader.exec_module(N)
contract=N.freeze_configuration(ROOT/'notebooks/59_new_sequential_confirmation.ipynb')
print('동결한 설정',contract['config_id'])

notebook59 cell 1
동결한 설정 old:A:A


위 셀의 설정 동결이 끝난 후에만 새로운 질문과 원문 평가 기준을 작성한다. 아래 셀은 질문별 원문 검토 및 파일 동결이 완료돼야 진행된다.

In [3]:
queries,contract,manifest,prompt=N.prepare(ROOT/'notebooks/59_new_sequential_confirmation.ipynb')
print('새 질문',len(queries),'조건 추출96회, 두 프로필 답변192회')

notebook59 cell 3
새 질문 32 조건 추출96회, 두 프로필 답변192회


In [4]:
resources=N.run(queries,contract,manifest,prompt,allow_network=True)
print(json.dumps(resources,ensure_ascii=False,indent=2))

notebook59 cell 4
59 retrieval 8 /192
59 retrieval 16 /192
59 retrieval 24 /192
59 retrieval 32 /192
`torch_dtype` is deprecated! Use `dtype` instead!
59 retrieval 40 /192
59 retrieval 48 /192
59 retrieval 56 /192
59 retrieval 64 /192
59 retrieval 72 /192
59 retrieval 80 /192
59 retrieval 88 /192
59 retrieval 96 /192
59 retrieval 104 /192
59 retrieval 112 /192
59 retrieval 120 /192
59 retrieval 128 /192
59 retrieval 136 /192
59 retrieval 144 /192
59 retrieval 152 /192
59 retrieval 160 /192
59 retrieval 168 /192
59 retrieval 176 /192
59 retrieval 184 /192
59 retrieval 192 /192
59 answer 8 / 192
59 answer 16 / 192
59 answer 24 / 192
59 answer 32 / 192
59 answer 40 / 192
59 answer 48 / 192
59 answer 56 / 192
59 answer 64 / 192
59 answer 72 / 192
59 answer 80 / 192
59 answer 88 / 192
59 answer 96 / 192
59 answer 104 / 192
59 answer 112 / 192
59 answer 120 / 192
59 answer 128 / 192
59 answer 136 / 192
59 answer 144 / 192
59 answer 152 / 192
59 answer 160 / 192
59 answer 168 / 192
59 answer 

## 실행 결과와 해석

32질문 × 2프로필 × 3회 = 192개 검색·답변 및 원문 수동 판정을 완료했다. 동일 질문·정답·추출 계획을 두 프로필에 사용했다. 최종 기술실패는 0건이다. 최종 독립 검토는 별도 receipt와 completion seal에서 확인한다.

| 지표 | benefit | parent-child |
|---|---:|---:|
| 중요조건 포함 질문 PASS (96답변) | 46/96 (47.92%) | 31/96 (32.29%) |
| PARTIAL / FAIL | 45 / 5 | 58 / 7 |
| 세 번 모두 PASS (32질문) | 12/32 | 8/32 |
| 유효 카드 Hit@3 (긍정 30질문×3회) | 87/90 (96.67%) | 87/90 (96.67%) |
| MRR@3 확인 하한 | 0.9444 | 0.9500 |
| 고정 참조 사실 검색·입력 회수율 (30질문 평균) | 59.61% | 49.41% |
| 입력된 고정 사실의 답변 반영률 (분모 존재 19질문) | 80.85% | 89.91% |

Hit@3는 실제 상위 3개 중 원문 확인된 유효 카드가 하나라도 있으면 성공, MRR@3는 첫 유효 카드 순위의 역수다. PASS는 실제 추천 카드의 중요한 실적·한도·적용조건까지 충족해야 한다. 고정 참조 목록은 전수 정답이 아니므로 유효한 대안 카드를 추천했을 때도 고정 참조 사실은 미검색으로 남을 수 있다. 고정 사실 지표를 실제 모든 카드의 근거 회수율로 해석하지 않는다. 기술실패와 부분 충족도 전체 PASS 분모에 포함한다. 지표별 예시·한계·분모·질문별 결과는 [59번 직접 작성 보고서](../docs/report/59_new_sequential_confirmation_report.html)에 정리했다.

실제 카드의 중요조건 누락 표식은 benefit 68개(검색·입력 30 / 생성 38), parent-child 106개(검색·입력 93 / 생성 13)였다. 이는 반복별 수동 표식의 건수이며 독립 사실 전체 회수율이나 모든 실패 원인의 전수 분류가 아니다. 카드 발견뿐 아니라 조건 관계 보존과 답변 반영을 개선해야 한다. 두 프로필 모두 카드 Hit는 높지만 최종 답변 완전성은 낮다.

## 비용·시간과 재현 한계

59번 실제 요청 292회(공유 추출 96, 임베딩 1, 답변 195) 모두 사용량이 확인됐다. 사용량과 동결 가격표 기준 추정 비용은 **$0.179209095**다. 실제 청구서 금액은 미측정이다. 46번 이후 누적 확인 비용은 $1.134396490이며 기존 미확인 두 건의 최대 예상액을 포함한 보수적 예산 차감은 **$1.140630990 / $8**이다.

답변 준비시간은 서비스 내부 검증·재시도 포함, benefit 평균 17.149초/P95 27.616초, parent-child 평균 16.393초/P95 26.132초다. 이후 전체 출처 확인·화면 출력 및 온라인 검색 전체 시간을 포함하지 않는다. TTFT·온라인 전체 응답시간·운영 SLA는 미측정이다. 조건 추출 평균은 7.056초이며 공유 처리 비용을 프로필에 임의 배분하지 않았다. 검색은 반복·프로필 사이의 재정렬 점수 캐시를 공유하므로 신규 질문의 최초 실행 시간으로 해석하지 않는다.

첫 실행은 종료 코드143으로 중단됐으며 원인은 미확인이다. 저장 결과를 검증해 재개했고, 위 코드 셀은 순서대로 실행하되 유효 캐시를 재사용했다. 처음부터 모든 API를 새로 호출한 완전 재실행은 아니다. 재개 실행만의 경과시간 90.41분, 최대 RSS 약3.062GiB를 기록했다. 최초 실행 전체 경과시간·CPU·메모리는 미측정이다.

192개 답변과 노트북 출력 저장 이후 임시 실행기의 마지막 자원 파일 수집에서 NameError가 발생했다. 종료 자원 기록으로 수집 파일만 복구했으며 이를 위해 API를 재호출하지 않았다. 이 코드 셀 자체나 동결 실험 코드를 수정하지 않았다. 세 답변은 인용 혼합·중복 카드·출력 토큰 한도로 인한 불완전 JSON을 같은 2400토큰 설정으로 재시도해 복구했으며 첫 시도도 비용·시간에 포함했다.

[질문별 전체 결과](data/59_new_sequential_confirmation/question_quality.json) · [원문·실제 입력·답변 판정](data/59_new_sequential_confirmation/source_judgments.json) · [사용량·자원 기록](data/59_new_sequential_confirmation/resource_summary.json)

운영/OCR 검증기/원본 DB·문서를 변경하지 않았다. 새 질문에 대한 비구조화 단독 기준선이 없어 현재 운영 대비 개선량은 계산하지 않는다. 무실적 복합문구, 카드·발급사 범위, 무료이용 동작, 대상 문자열과 혜택 관계에 남은 한계를 후속 수정·회귀 대상으로 기록했다.
